<a href="https://colab.research.google.com/github/hubdk17/OpenCV-/blob/code/CVproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# train_dogbreed_resnet.py

import kagglehub
import os
import torch
import torchvision
from torch import nn, optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

# --- DOWNLOAD DATA ---
base_path = kagglehub.dataset_download("kabilan03/dogbreedclassification")
print("Dataset base path:", base_path)

DATA_DIR = os.path.join(base_path, "Dog Breed Classification")

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR   = os.path.join(DATA_DIR, "val")
TEST_DIR  = os.path.join(DATA_DIR, "test")

# Hyperparameters
BATCH_SIZE = 32
LR         = 1e-4
EPOCHS     = 12

# --- TRANSFORMS ---
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# --- DATASETS ---
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_ds   = datasets.ImageFolder(VAL_DIR,   transform=val_test_transform)
test_ds  = datasets.ImageFolder(TEST_DIR,  transform=val_test_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

# --- MODEL ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(train_ds.classes)  # 93 classes
model = torchvision.models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# --- TRAIN ---
for epoch in range(EPOCHS):
    model.train()
    running_loss, running_correct = 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_correct += (outputs.argmax(1) == labels).sum().item()

    train_acc = running_correct / len(train_ds)
    print(f"[Epoch {epoch+1}] Loss: {running_loss:.4f}, Train Acc: {train_acc:.4f}")

# Save model
torch.save(model.state_dict(), "dogbreed_resnet50.pth")
print("Training complete!")

100%|██████████| 272M/272M [00:13<00:00, 21.4MB/s]

Extracting files...


Dataset base path: /root/.cache/kagglehub/datasets/kabilan03/dogbreedclassification/versions/3


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 143MB/s]


[Epoch 1] Loss: 422.3530, Train Acc: 0.5735
[Epoch 2] Loss: 130.8072, Train Acc: 0.8488
[Epoch 3] Loss: 70.1080, Train Acc: 0.9169
[Epoch 4] Loss: 45.6041, Train Acc: 0.9468
[Epoch 5] Loss: 29.3205, Train Acc: 0.9646
[Epoch 6] Loss: 22.4719, Train Acc: 0.9750
[Epoch 7] Loss: 21.6847, Train Acc: 0.9747
[Epoch 8] Loss: 18.7003, Train Acc: 0.9781
[Epoch 9] Loss: 16.9503, Train Acc: 0.9789
[Epoch 10] Loss: 14.6080, Train Acc: 0.9826
[Epoch 11] Loss: 17.2721, Train Acc: 0.9778
[Epoch 12] Loss: 16.7784, Train Acc: 0.9789
Training complete!


# Dog Breed Classification with ResNet50

This project implements a dog breed classification model using a pre-trained ResNet50 convolutional neural network. The model is fine-tuned on a custom dataset of dog breeds and evaluated on a validation set. Additionally, it demonstrates how to use a pre-trained ResNet50 (on ImageNet) for general image classification using OpenCV.

## Project Structure

The project consists of the following main steps:

1.  **Data Download**: Downloads the dog breed classification dataset from KaggleHub.
2.  **Data Preparation**: Defines image transformations and creates data loaders for training and validation.
3.  **Model Definition**: Initializes a pre-trained ResNet50 model and modifies its final classification layer to match the number of dog breeds for transfer learning.
4.  **Training**: Trains (fine-tunes) the ResNet50 model using Cross-Entropy Loss and the Adam optimizer on the custom dataset.
5.  **Evaluation**: Evaluates the fine-tuned model on a validation set, reporting accuracy, precision, recall, and F1-score.
6.  **Breed Identification (using general ImageNet ResNet50)**: Demonstrates image classification using a standard ImageNet pre-trained ResNet50 and `cv2` for image processing.

## Dataset

The dataset used for fine-tuning this project is hosted on KaggleHub: `kabilan03/dogbreedclassification`.
It is structured into `train`, `val`, and `test` directories, each containing subdirectories for different dog breeds.

## Model

The primary model used is `ResNet50`, pre-trained on ImageNet. For the dog breed classification task, the final fully connected layer is replaced to classify 93 distinct dog breeds (transfer learning). A separate instance of the ImageNet pre-trained ResNet50 is also used for general image classification demonstration.

## Setup and Installation

To run this code, you need to have Python and the following libraries installed:

*   `kagglehub`
*   `torch`
*   `torchvision`
*   `scikit-learn` (for evaluation metrics)
*   `opencv-python` (for image processing in breed identification)

You can install them using pip:

```bash
pip install kagglehub torch torchvision scikit-learn opencv-python
```

## Usage

### 1. Download Data

The dataset is automatically downloaded to a local cache directory using `kagglehub.dataset_download`.

```python
import kagglehub
base_path = kagglehub.dataset_download("kabilan03/dogbreedclassification")
```

### 2. Train the Model (Transfer Learning)

Run the training cell (`CGjKnssbdVJa`) in the notebook. This will:

*   Load the dataset.
*   Apply image transformations (resize, random horizontal flip, normalization).
*   Initialize a pre-trained ResNet50 model and fine-tune its classification layer for the specific dog breeds for `EPOCHS` (default: 12) epochs.
*   Save the trained model's state dictionary to `dogbreed_resnet50.pth`.

```python
# Snippet from training cell
# ...
model = torchvision.models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
# ...
for epoch in range(EPOCHS):
    # Training loop
    # ...
torch.save(model.state_dict(), "dogbreed_resnet50.pth")
print("Training complete!")
```

### 3. Evaluate the Fine-tuned Model

Run the evaluation cell (`ycu2K0apS_LC`) in the notebook. This will:

*   Load the validation dataset.
*   Load the previously saved fine-tuned model (`dogbreed_resnet50.pth`).
*   Perform inference on the validation set.
*   Calculate and print accuracy, precision, recall, and F1-score.

```python
# Snippet from evaluation cell
# ...
model.load_state_dict(torch.load("dogbreed_resnet50.pth", map_location=device))
# ...
# Compute metrics and print results
```

### 4. Breed Identification (using `cv2` and ImageNet pre-trained ResNet50)

Run the cell (`U6xIFjdTty-w`) that uses `cv2` to load and process an image. This demonstrates a general image classification using a standard ResNet50 model pre-trained on ImageNet. This part of the code:

*   Loads a pre-trained ResNet-50 model from `torchvision` (pre-trained on ImageNet).
*   Downloads ImageNet labels.
*   Reads a sample image (`sample_dog.jpg`) using `cv2` and converts it to RGB.
*   Applies standard ImageNet preprocessing transformations.
*   Performs inference to predict the image's category and confidence.

```python
# Snippet from breed identification cell
# ...
import cv2
from PIL import Image
# ...
model = models.resnet50(weights='IMAGENET1K_V1') # Loads ImageNet pre-trained weights
# ...
image_cv = cv2.imread(img_path)
image_rgb = cv2.cvtColor(image_cv, cv2.COLOR_BGR2RGB)
# ...
# Prediction and printing results
print(f"Prediction: {labels[category_id]}")
print(f"Confidence: {confidence:.2f}%")
```

## Results

After fine-tuning for 12 epochs, the model achieved the following performance on the *custom dog breed validation set*:

*   **Validation Accuracy :** 0.7493
*   **Validation Precision:** 0.7826
*   **Validation Recall   :** 0.7493
*   **Validation F1 Score :** 0.7412

These metrics indicate a reasonably good performance for classifying 93 different dog breeds. The separate demonstration using a general ImageNet pre-trained ResNet50 for 'sample_dog.jpg' yielded:

*   **Prediction:** pug
*   **Confidence:** 88.96%

In [7]:
print(base_path)
!ls "{base_path}"

/root/.cache/kagglehub/datasets/kabilan03/dogbreedclassification/versions/3
'Dog Breed Classification'


In [8]:
# evaluate_on_val_set.py

import os
import torch
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Re-use the same base path from your training cell
base_path = base_path  # example: '/root/.cache/kagglehub/datasets/kabilan03/dogbreedclassification/versions/3'

# Construct path to the validation set
VAL_DIR = os.path.join(base_path, "Dog Breed Classification", "val")

# Image transforms: must match training normalization
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

# Load validation dataset
val_ds = datasets.ImageFolder(VAL_DIR, transform=transform)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load your trained ResNet50 model
model = torchvision.models.resnet50(pretrained=False)
model.fc = torch.nn.Linear(model.fc.in_features, len(val_ds.classes))
model.load_state_dict(torch.load("dogbreed_resnet50.pth", map_location=device))
model.to(device)
model.eval()

# Collect predictions and true labels
y_true = []
y_pred = []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(labels.numpy())

# Compute metrics
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

print(f"Validation Accuracy : {acc:.4f}")
print(f"Validation Precision: {prec:.4f}")
print(f"Validation Recall   : {rec:.4f}")
print(f"Validation F1 Score : {f1:.4f}")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Validation Accuracy : 0.7493
Validation Precision: 0.7826
Validation Recall   : 0.7493
Validation F1 Score : 0.7412


In [9]:
import cv2
import torch
import requests
from torchvision import models, transforms
from PIL import Image
import numpy as np

# 1. Load the Pre-trained ResNet-50 Model
model = models.resnet50(weights='IMAGENET1K_V1')
model.eval()

# 2. Download ImageNet labels (to turn numbers into breed names)
LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = requests.get(LABELS_URL).text.splitlines()

# 3. Open image using OpenCV and convert BGR to RGB
img_path = 'sample_dog.jpg'
image_cv = cv2.imread(img_path)
image_rgb = cv2.cvtColor(image_cv, cv2.COLOR_BGR2RGB)

# 4. Prepare the image for ResNet
# (ResNet needs 224x224, Tensor format, and specific Normalization)
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Convert the OpenCV image (numpy) to PIL for the transform
image_pil = Image.fromarray(image_rgb)
input_tensor = preprocess(image_pil)
input_batch = input_tensor.unsqueeze(0) # Add a batch dimension

# 5. Prediction
if torch.cuda.is_available():
    input_batch = input_batch.to('cuda')
    model.to('cuda')

with torch.no_grad():
    output = model(input_batch)

# 6. Get the result
probabilities = torch.nn.functional.softmax(output[0], dim=0)
category_id = torch.argmax(probabilities).item()
confidence = probabilities[category_id].item() * 100

print(f"Prediction: {labels[category_id]}")
print(f"Confidence: {confidence:.2f}%")

Prediction: pug
Confidence: 88.96%
